# Custom Face Recognition Model Training (FYP)

This notebook covers the complete pipeline for training a lightweight face recognition model optimized for edge devices (like a Raspberry Pi). We will:
1. Load the Nepali Actors dataset.
2. Perform Exploratory Data Analysis (Class Distributions & Samples).
3. Set up **Image Augmentation** (cropping, flipping, color jitter).
4. Construct a model using a lightweight backbone (`MobileNetV2`) and **ArcFace** metric learning loss.
5. Train the network to generate 512-dimensional face embeddings.
6. **Export the architecture to ONNX** format so it can be swapped into our InsightFace smart mirror app.

In [ ]:
import os
import glob
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms, datasets, models

# Configuration
DATA_DIR = 'Dataset/Dataset'
ACTORS_FILE = 'actors.txt'
BATCH_SIZE = 32
EPOCHS = 45 # Increased epochs since we lowered learning rate
EMBEDDING_SIZE = 512
IMAGE_SIZE = 112  # InsightFace standard input size

print(f"CUDA Available: {torch.cuda.is_available()}")

## 1. Exploratory Data Analysis (EDA)
Let's look at the dataset class distribution and list of actors.

In [ ]:
# Load Actors
if os.path.exists(ACTORS_FILE):
    with open(ACTORS_FILE, 'r', encoding='utf-8') as f:
        actors = [line.strip() for line in f.readlines() if line.strip()]
    print(f"Loaded {len(actors)} actors from {ACTORS_FILE}")
else:
    print(f"Warning: {ACTORS_FILE} not found!")

# Analyze directory distributions
class_counts = {}
if os.path.exists(DATA_DIR):
    for person_name in os.listdir(DATA_DIR):
        person_path = os.path.join(DATA_DIR, person_name)
        if os.path.isdir(person_path):
            # count images
            count = len(glob.glob(os.path.join(person_path, '*.*')))
            class_counts[person_name] = count

df_counts = pd.DataFrame(list(class_counts.items()), columns=['Actor', 'Image_Count'])
df_counts = df_counts.sort_values(by='Image_Count', ascending=False)

plt.figure(figsize=(16, 7))
sns.barplot(data=df_counts.head(40), x='Actor', y='Image_Count', palette='viridis')
plt.xticks(rotation=90)
plt.title('Top 40 Actors by Image Count')
plt.tight_layout()
plt.show()

print(f"\nTotal distinct classes (actors) found: {len(class_counts)}")
print(f"Total images across dataset: {df_counts['Image_Count'].sum()}")

In [ ]:
# Visualize Sample Images
samples_to_show = 6
fig, axes = plt.subplots(1, samples_to_show, figsize=(18, 4))
top_identities = df_counts['Actor'].head(samples_to_show).values

for i, identity in enumerate(top_identities):
    identity_path = os.path.join(DATA_DIR, identity)
    img_paths = glob.glob(os.path.join(identity_path, '*.*'))
    if img_paths:
        img = Image.open(img_paths[0]).convert('RGB')
        axes[i].imshow(img)
        axes[i].set_title(identity, fontsize=10)
        axes[i].axis('off')

plt.suptitle("Sample Face Images from Dataset", fontsize=16, y=1.05)
plt.show()

## 2. Setting Up Augmentations and DataLoaders

In [ ]:
# Define Transforms for Training (with Augmentation) and Validation
train_transform = transforms.Compose([
    transforms.Resize((128, 128)),            # Resize slightly larger
    transforms.RandomCrop(IMAGE_SIZE),        # Crop to exactly 112x112
    transforms.RandomHorizontalFlip(p=0.5),   # Mirroring
    transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.15), # Milder lighting changes
    transforms.RandomRotation(degrees=7),     # Slight tilts (reduced to maintain facial alignment)
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]) # Standard format for models
])

val_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)), # Strict resize to 112x112
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

# Custom Dataset Wrapper to apply different transforms to train/val splits
class DualTransformDataset(Dataset):
    def __init__(self, subset, transform=None):
        self.subset = subset
        self.transform = transform

    def __getitem__(self, index):
        x, y = self.subset[index]
        if self.transform:
            x = self.transform(x)
        return x, y

    def __len__(self):
        return len(self.subset)

# Load full images
try:
    full_dataset = datasets.ImageFolder(root=DATA_DIR)
    num_classes = len(full_dataset.classes)

    # Split 80/20 train/validation
    train_size = int(0.8 * len(full_dataset))
    val_size = len(full_dataset) - train_size
    train_subset, val_subset = random_split(full_dataset, [train_size, val_size])

    # Overwrite the transform internally to block the original one
    train_subset.dataset.transform = None 

    train_data = DualTransformDataset(train_subset, transform=train_transform)
    val_data = DualTransformDataset(val_subset, transform=val_transform)

    # Create Loaders
    train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
    val_loader = DataLoader(val_data, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

    print(f"Data loaded! Classes: {num_classes} | Train samples: {train_size} | Val samples: {val_size}")
except Exception as e:
    print("Error loading dataset. Make sure Dataset/Dataset exists and has folders inside it.")
    print(str(e))

## 3. Define the Architecture (ArcFace + MobileFaceNet)

We use **ArcFace** (Additive Angular Margin Loss) instead of standard Cross-Entropy. This trains the model to cluster faces in a metric space, so we can use Cosine Similarity to compare them later.

In [ ]:
class CosineLinearHead(nn.Module):
    """
    A simpler, highly stable metric learning head. 
    Instead of ArcFace's aggressive margins (which destroy small datasets), 
    this computes pure Cosine Similarity and scales it.
    This forces embeddings onto a hypersphere perfectly suited for your edge app!
    """
    def __init__(self, in_features, out_features, s=20.0):
        super(CosineLinearHead, self).__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.s = s
        self.weight = nn.Parameter(torch.FloatTensor(out_features, in_features))
        nn.init.xavier_uniform_(self.weight)

    def forward(self, input, label):
        # Pure cosine similarity between embeddings and class centers
        cosine = F.linear(F.normalize(input), F.normalize(self.weight))
        output = cosine * self.s
        return output

class LightweightFaceModel(nn.Module):
    """
    A lightweight backbone utilizing MobileNetV2, re-headed to output a 512-dimension embedding.
    """
    def __init__(self, embedding_size=512):
        super(LightweightFaceModel, self).__init__()
        self.backbone = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.DEFAULT)
        
        # Remove default 1000-class classifier
        self.backbone.classifier = nn.Identity()
        
        # Add projection to 512 dimensions with BatchNorm and PReLU
        self.projection = nn.Sequential(
            nn.Linear(1280, embedding_size),
            nn.BatchNorm1d(embedding_size),
            nn.PReLU(), # PReLU works phenomenally well in Face Recognition vs ReLU
            nn.Dropout(0.3) 
        )

    def forward(self, x):
        features = self.backbone(x)
        embeddings = self.projection(features)
        return embeddings

# Setup Model & Optimizer
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[*] Training on device: {device}")

if 'num_classes' in locals():
    model = LightweightFaceModel(embedding_size=EMBEDDING_SIZE).to(device)
    metric_fc = CosineLinearHead(in_features=EMBEDDING_SIZE, out_features=num_classes).to(device)

    # 1. LABEL SMOOTHING: Prevents overconfidence on specific pixels. Excellent for small datasets!
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    
    # 2. PARTIAL LAYER FREEZING: Pre-trained MobileNetV2 already knows how to see edges, noses, eyes.
    # We freeze the first 10 blocks so those visual feature filters aren't destroyed.
    for param in model.backbone.features[:10].parameters():
        param.requires_grad = False
    
    # 3. STRATEGIC LEARNING RATES & HEAVY WEIGHT DECAY
    optimizer = optim.AdamW([
        {'params': model.backbone.features[10:].parameters(), 'lr': 1e-4}, # Slow fine-tuning for deep layers
        {'params': model.projection.parameters(), 'lr': 5e-4},             # Moderate learning for new head
        {'params': metric_fc.parameters(), 'lr': 5e-4}                     # Moderate learning for metric centers
    ], weight_decay=0.05) # Increased weight decay forces the model to use the whole face, not just 1 feature

## 4. The Training Loop
Run this to fine-tune the model.

In [ ]:
best_val_loss = float('inf')
loss_history = []

if 'num_classes' in locals() and num_classes > 0:
    print("Starting Training...")
    
    # 4. COSINE ANNEALING SCHEDULER: Instead of reacting to plateaus, proactively surf down a smooth mathematical wave.
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)

    for epoch in range(EPOCHS):
        # --- Train Phase ---
        model.train()
        metric_fc.train()
        running_loss = 0.0
        
        for i, (images, labels) in enumerate(train_loader):
            images, labels = images.to(device), labels.to(device)
            
            optimizer.zero_grad()
            
            embeddings = model(images)
            outputs = metric_fc(embeddings, labels)
            
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item()
            
        avg_train_loss = running_loss / len(train_loader)
        
        # --- Validation Phase ---
        model.eval()
        metric_fc.eval()
        val_loss = 0.0
        correct = 0
        total = 0
        
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                
                embeddings = model(images)
                outputs = metric_fc(embeddings, labels)
                
                loss = criterion(outputs, labels)
                val_loss += loss.item()
                
                _, predicted = torch.max(outputs.data, 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()
                
        avg_val_loss = val_loss / len(val_loader)
        val_acc = 100.0 * correct / total
        loss_history.append((avg_train_loss, avg_val_loss, val_acc))
        
        # Step the scheduler (CosineAnnealingLR steps continuously instead of needing val_loss)
        scheduler.step()
        
        current_lr = optimizer.param_groups[-1]['lr']
        print(f"Epoch {epoch+1:02d}/{EPOCHS} | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | Val Acc: {val_acc:.2f}% | LR: {current_lr:.6f}")
        
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            torch.save(model.state_dict(), 'best_nepali_face_model.pth')
            print("  -> Saved new best model!")

    print("Training Complete!")
else:
    print("dataset is not loaded. Skipping training loop.")

In [ ]:
# Plotting the Metrics
if loss_history:
    train_losses = [x[0] for x in loss_history]
    val_losses = [x[1] for x in loss_history]

    plt.figure(figsize=(10, 5))
    plt.plot(train_losses, label='Train Loss', color='blue')
    plt.plot(val_losses, label='Validation Loss', color='orange')
    plt.title('Training & Validation Loss over Epochs')
    plt.xlabel('Epochs')
    plt.ylabel('Loss Value')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()

## 5. Export to ONNX 
InsightFace uses ONNX (`.onnx`) for maximum, hyper-optimized performance on CPUs (like the Raspberry Pi). This cell exports your trained PyTorch weights so they can be dropped right into your `simple_server.py` implementation.

In [ ]:
if os.path.exists('best_nepali_face_model.pth'):
    model.load_state_dict(torch.load('best_nepali_face_model.pth', map_location=device))
    model.eval()

    # Create a dummy input matching the shape of a single image (1, 3, 112, 112)
    dummy_input = torch.randn(1, 3, 112, 112).to(device)
    onnx_file_path = "nepali_mbf.onnx"

    # Export the architecture
    torch.onnx.export(
        model, 
        dummy_input, 
        onnx_file_path, 
        export_params=True, 
        opset_version=11,          
        do_constant_folding=True,  
        input_names=['input'],     
        output_names=['output'],   
        dynamic_axes={'input': {0: 'batch_size'}, 'output': {0: 'batch_size'}}
    )

    print(f"\n[SUCCESS] Model exported to {onnx_file_path}")
    print("To use it in your app:")
    print("1. Put this file inside a folder named '~/.insightface/models/nepali_pi/'")
    print("2. Put the 'det_10g.onnx' (or det_500m.onnx) face detector file into that same folder.")
    print("3. Update simple_server.py line to:  FaceAnalysis(name='nepali_pi')")
else:
    print("Could not find best_nepali_face_model.pth. Train the model first!")